# Binary Decomposition – Experiment Dashboard

Postup experimentov:
1. **Prehľad datasetov** – vizualizácia a štatistiky vstupných dát
2. **Analýza hyperparametrov** (`analysis/objects_unique`, `analysis/leafs_subset`) – hľadanie optimálnych parametrov
   - EXP-1: pop_size a patience
   - EXP-2: Inicializačná metóda GA
   - EXP-3: Metóda crossoveru
   - EXP-4: Mutačné pravdepodobnosti (p_local, p_merge)
3. **Full Run** (`leafs_selected`, `objects_selected`) – finálne porovnanie algoritmov

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
from adjustText import adjust_text

# ── Paths ────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('../../..')  # from notebooks/dashboard/
CSV_DIR  = PROJECT_ROOT / 'experiments/results/csv'
DATA_DIR = PROJECT_ROOT / 'data/datasets'

# ── Publication-ready style ──────────────────────────────────────────
plt.rcParams.update({
    # Background
    'figure.facecolor':      'white',
    'axes.facecolor':        'white',
    'savefig.facecolor':     'white',
    'savefig.edgecolor':     'white',
    # Text colors — explicit dark so nothing disappears on white
    'text.color':            '#222222',
    'axes.labelcolor':       '#222222',
    'axes.titlecolor':       '#222222',
    'xtick.color':           '#444444',
    'ytick.color':           '#444444',
    'xtick.labelcolor':      '#222222',
    'ytick.labelcolor':      '#222222',
    'legend.labelcolor':     '#222222',
    # Grid
    'axes.grid':             True,
    'grid.color':            '#e0e0e0',
    'grid.linewidth':        0.8,
    'axes.axisbelow':        True,
    # Spines
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.spines.left':      True,
    'axes.spines.bottom':    True,
    'axes.edgecolor':        '#444444',
    'axes.linewidth':        0.8,
    # Font
    'font.family':           'sans-serif',
    'font.size':             11,
    'axes.titlesize':        13,
    'axes.labelsize':        11,
    'legend.fontsize':       10,
    'legend.framealpha':     0.9,
    'legend.edgecolor':      '#cccccc',
    'legend.facecolor':      'white',
    # Lines
    'lines.linewidth':       2.0,
    'lines.antialiased':     True,
    # Output quality
    'figure.dpi':            150,
    'savefig.dpi':           300,
    'savefig.bbox':          'tight',
    'savefig.pad_inches':    0.1,
})

COLORS = plt.cm.Set2.colors

In [ ]:
# ── Helper: load all results CSVs for a given algorithm ──────────────
def load_results(algorithm: str, dataset: str, run_id: str = None) -> pd.DataFrame:
    """Load results CSV for an algorithm/dataset combination.
    
    If run_id is None, loads and concatenates all run subdirectories.
    """
    base = CSV_DIR / algorithm
    if not base.exists():
        return pd.DataFrame()
    
    dfs = []
    pattern = f"{dataset}/{run_id}/results.csv" if run_id else f"{dataset}/**/results.csv"
    for csv_path in sorted(base.glob(pattern)):
        try:
            df = pd.read_csv(csv_path)
            df['algorithm'] = algorithm
            df['dataset'] = dataset
            df['run_path'] = str(csv_path.parent.relative_to(base))
            dfs.append(df)
        except Exception:
            pass
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def load_generations(algorithm: str, dataset: str) -> pd.DataFrame:
    """Load generation history CSVs for a GA algorithm."""
    base = CSV_DIR / algorithm
    if not base.exists():
        return pd.DataFrame()
    dfs = []
    for csv_path in sorted(base.glob(f"{dataset}/**/generations.csv")):
        try:
            df = pd.read_csv(csv_path)
            df['algorithm'] = algorithm
            df['run_path'] = str(csv_path.parent.relative_to(base))
            dfs.append(df)
        except Exception:
            pass
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print('Helpers loaded.')

In [ ]:
DETERMINISTIC_ALGOS = ['dm', 'gdm', 'quadtree', 'largest_rect', 'graph_based']
GA_ALGOS = ['ga_dm', 'ga_gdm', 'ga_random', 'ga_qtd', 'ga_lrf']
DATASETS_COMPARE = ['leafs_selected', 'objects_selected']

def load_baseline(algo: str, ds: str) -> pd.DataFrame:
    """Load baseline run — any run_id containing 'run1' but not 'run2'.

    For quadtree, run2 is coverage-only (not full decomposition) and must
    never be used as a baseline.
    """
    df = load_results(algo, ds)
    if df.empty:
        return df
    mask = (
        ~df['run_path'].str.contains('test|exp2|exp3|run2', na=False)
        & df['run_path'].str.contains('run1', na=False)
    )
    return df[mask].drop_duplicates(subset='image_name')

def load_ga_baseline(algo: str, ds: str) -> pd.DataFrame:
    """Load GA results — non-test/exp, deduplicated per image."""
    df = load_results(algo, ds)
    if df.empty:
        return df
    df = df[~df['run_path'].str.contains('test|exp2|exp3', na=False)]
    return df.sort_values('rectangle_count').drop_duplicates(subset='image_name')


# ── Monitor config ────────────────────────────────────────────────────
# Formát: algo → [(run_id, display_label), ...]
# Viac run_ids pre jeden algoritmus = všetky sa sledujú v tabuľke.
MONITOR_CONFIG = {
    'dm':           [('run1', 'dm')],
    'gdm':          [('run1', 'gdm')],
    'quadtree':     [('run1', 'quadtree_fd'), ('run2', 'quadtree_cov')],
    'largest_rect': [('run1', 'largest_rect')],
    'graph_based':  [('run1', 'graph_based')],
    # GA — pridaj/odober run_ids podľa potreby
    'ga_dm': [
        # ('run1',             'ga_dm/run1'),
        ('run2-pop15-l6-m5', 'ga_dm'),
    ],
    'ga_gdm': [
        # ('run1',             'ga_gdm/run1'),
        ('run2',             'ga_gdm'),
    ],
    'ga_random': [],
    'ga_qtd':    [],
    'ga_lrf':    [],
}


def _load_run_seeded(algo: str, ds: str, run_id: str) -> pd.DataFrame:
    """GA: načíta <run_id>/seed_*/results.csv a pridá stĺpec seed."""
    base = CSV_DIR / algo / ds / run_id
    if not base.exists():
        return pd.DataFrame()
    dfs = []
    for p in sorted(base.glob('seed_*/results.csv')):
        try:
            df = pd.read_csv(p)
            df['seed'] = p.parent.name
            dfs.append(df)
        except Exception:
            pass
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def load_monitored(algo: str, ds: str, run_id: str) -> pd.DataFrame:
    """Načíta výsledky pre monitorovanie. GA: best result per image across seeds."""
    if algo not in GA_ALGOS:
        p = CSV_DIR / algo / ds / run_id / 'results.csv'
        return pd.read_csv(p) if p.exists() else pd.DataFrame()
    df = _load_run_seeded(algo, ds, run_id)
    if df.empty:
        return df
    return df.sort_values('rectangle_count').drop_duplicates(subset='image_name')


def _dataset_total(ds: str) -> int:
    m = DATA_DIR / ds / 'manifest.csv'
    if m.exists():
        return len(pd.read_csv(m))
    npy = DATA_DIR / ds / 'npy'
    return sum(1 for _ in npy.glob('*.npy')) if npy.exists() else 0


print('Baseline helpers and monitor config loaded.')


## 1. Prehľad datasetov

In [ ]:
# Load manifests if available, otherwise scan npy dirs
def dataset_info(name: str) -> pd.DataFrame:
    manifest = DATA_DIR / name / 'manifest.csv'
    if manifest.exists():
        return pd.read_csv(manifest)
    # Fall back: scan npy dir for shapes
    npy_dir = DATA_DIR / name / 'npy'
    rows = []
    for p in sorted(npy_dir.glob('*.npy')):
        arr = np.load(p, mmap_mode='r')
        h, w = arr.shape[0], arr.shape[1]
        rows.append({'image_name': p.name, 'height': h, 'width': w, 'pixels': h * w})
    return pd.DataFrame(rows)

leafs_info = dataset_info('leafs_selected')
objects_info = dataset_info('objects_selected') if (DATA_DIR / 'objects_selected').exists() else pd.DataFrame()

print(f"Leafs:   {len(leafs_info)} images, "
      f"pixels {leafs_info['pixels'].min():,} – {leafs_info['pixels'].max():,}")
if not objects_info.empty:
    print(f"Objects: {len(objects_info)} images, "
          f"pixels {objects_info['pixels'].min():,} – {objects_info['pixels'].max():,}")

In [ ]:
for info, title, fname in [
    (leafs_info,   'Listy (leafs_selected)',     'fig_01a_dataset_leafs.png'),
    (objects_info, 'Objekty (objects_selected)', 'fig_01b_dataset_objects.png'),
]:
    if info.empty:
        print(f'No data for {title}')
        continue
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(info['pixels'] / 1000, bins=30, color='steelblue', edgecolor='white')
    ax.set_xlabel('Veľkosť obrázka (kpx)')
    ax.set_ylabel('Počet obrázkov')
    plt.tight_layout()
    plt.savefig(fname, bbox_inches='tight')
    plt.show()

## 2. Analýza hyperparametrov

Všetky experimenty tejto sekcie sú spúšťané na **analysis datasetoch**
(`objects_unique`, `leafs_subset`, max 10 obrázkov).
Výsledky určujú finálnu konfiguráciu pre full run.

### EXP-1A – Vplyv pop_size (ga_dm, analysis datasety)

In [ ]:
pop_records = []
for ds in ['analysis/objects_quartile', 'analysis/leafs_quartile']:
    df = load_results('ga_dm', ds)
    if df.empty:
        continue
    exp1 = df[df['run_path'].str.contains('exp2_popsize|exp2_patience', na=False)].copy()
    if exp1.empty:
        continue
    exp1['sweep'] = exp1['run_path'].str.extract(r'(exp2_\w+)')
    exp1['value'] = exp1['run_path'].str.extract(r'exp2_\w+/(\d+)').astype(float)
    for _, row in exp1.iterrows():
        pop_records.append({
            'sweep': row['sweep'],
            'value': row['value'],
            'dataset': ds,
            'image_name': row['image_name'],
            'rectangle_count': row['rectangle_count'],
            'execution_time_sec': row['execution_time_sec'],
            'generations_used': row.get('generations_used'),
        })

pop_df = pd.DataFrame(pop_records)
if not pop_df.empty:
    for sweep in ['exp2_popsize', 'exp2_patience']:
        sub = pop_df[pop_df['sweep'] == sweep]
        if sub.empty:
            continue
        print(f'\n=== {sweep} ===')
        display(sub.groupby(['value', 'dataset'])
                [['rectangle_count', 'execution_time_sec']]
                .agg(['mean', 'std']).round(2))
else:
    print('EXP-1 results not available yet.')
    print('Run: python -m experiments.scripts.analysis.run_ga_exp1_dm_popsize')

In [ ]:
pop_size_df = pop_df[pop_df['sweep'] == 'exp2_popsize'] if not pop_df.empty else pd.DataFrame()

if not pop_size_df.empty:
    ds_list = pop_size_df['dataset'].unique()
    fig, axes = plt.subplots(1, len(ds_list),
                             figsize=(6 * len(ds_list), 6),
                             squeeze=False)

    for col_i, ds in enumerate(ds_list):
        ax = axes[0][col_i]
        sub = pop_size_df[pop_size_df['dataset'] == ds].copy()
        if sub.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue

        min_val = sub['value'].min()
        baseline = (sub[sub['value'] == min_val]
                    .set_index('image_name')['rectangle_count']
                    .rename('baseline'))
        sub = sub.join(baseline, on='image_name')
        sub['rect_norm'] = sub['rectangle_count'] / sub['baseline']

        stats = sub.groupby('value')['rect_norm'].agg(['mean', 'std']).reset_index()
        ax.plot(stats['value'], stats['mean'], 'o-',
                color='steelblue', linewidth=2, markersize=7)
        ax.fill_between(stats['value'],
                        stats['mean'] - stats['std'],
                        stats['mean'] + stats['std'],
                        alpha=0.15, color='steelblue')
        ax.axhline(1.0, color='gray', linestyle='--', linewidth=1)
        ax.axvline(15, color='red', linestyle='--', linewidth=1.5, label='zvolené pop_size=15')
        ax.legend(fontsize=11)
        ax.set_xlabel('Veľkosť populácie', fontsize=13)
        ax.set_ylabel(f'Norm. počet obdĺžnikov (základ: pop_size={int(min_val)})', fontsize=13)
        ax.tick_params(axis='both', labelsize=12)
        ax.set_xticks(stats['value'].tolist())

    plt.tight_layout()
    plt.savefig('fig_exp1a_popsize.png', bbox_inches='tight')
    plt.show()
else:
    print('EXP-2A results not available yet.')
    print('Run: python -m experiments.scripts.analysis.run_ga_exp2_popsize')

In [ ]:
patience_sweep_df = pop_df[pop_df['sweep'] == 'exp2_patience'] if not pop_df.empty else pd.DataFrame()

if not patience_sweep_df.empty:
    ds_list = patience_sweep_df['dataset'].unique()
    fig, axes = plt.subplots(1, len(ds_list),
                             figsize=(6 * len(ds_list), 6),
                             squeeze=False)

    for col_i, ds in enumerate(ds_list):
        ax = axes[0][col_i]
        sub = patience_sweep_df[patience_sweep_df['dataset'] == ds].copy()
        if sub.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue

        # Normalize per image: divide each image's rect_count by its value at min patience
        min_patience = sub['value'].min()
        baseline = (sub[sub['value'] == min_patience]
                    .set_index('image_name')['rectangle_count']
                    .rename('baseline'))
        sub = sub.join(baseline, on='image_name')
        sub['rect_norm'] = sub['rectangle_count'] / sub['baseline']

        stats = sub.groupby('value')['rect_norm'].agg(['mean', 'std']).reset_index()
        ax.plot(stats['value'], stats['mean'], 'o-',
                color='steelblue', linewidth=2, markersize=7)
        ax.fill_between(stats['value'],
                        stats['mean'] - stats['std'],
                        stats['mean'] + stats['std'],
                        alpha=0.15, color='steelblue')
        ax.axhline(1.0, color='gray', linestyle='--', linewidth=1)
        ax.axvline(12, color='red', linestyle='--', linewidth=1.5, label='zvolené patience=12')
        ax.legend(fontsize=11)
        ax.set_xlabel('Stagnačný limit', fontsize=13)
        ax.set_ylabel(f'Norm. počet obdĺžnikov (základ: patience={int(min_patience)})', fontsize=13)
        ax.tick_params(axis='both', labelsize=12)
        ax.set_xticks(stats['value'].tolist())

    plt.tight_layout()
    plt.savefig('fig_exp1b_patience.png', bbox_inches='tight')
    plt.show()
else:
    print('EXP-2B sweep results not available yet.')
    print('Run: python -m experiments.scripts.analysis.run_ga_exp2_popsize')

### EXP-2B – Analýza patience

Hľadáme minimálnu hodnotu patience kde GA stále konverguje.
Analýza z `generations.csv` dát — proxy cez `best_rectangle_count`.

In [ ]:
patience_sweep_df = pop_df[pop_df['sweep'] == 'exp2_patience'] if not pop_df.empty else pd.DataFrame()

if not patience_sweep_df.empty:
    ds_list = sorted(patience_sweep_df['dataset'].str.replace('analysis/', '', regex=False).unique())
    patience_vals = sorted(patience_sweep_df['value'].unique())

    print('Dáta pre plot (mean rect_count per patience per dataset):')
    display(patience_sweep_df.groupby(['value', 'dataset'])['rectangle_count']
            .agg(['mean', 'std', 'count']).round(2))

    fig, axes = plt.subplots(1, len(ds_list), figsize=(6 * len(ds_list), 5), squeeze=False)
    for col_i, ds in enumerate(ds_list):
        ax = axes[0][col_i]
        sub = patience_sweep_df[patience_sweep_df['dataset'].str.endswith(ds)]
        stats = sub.groupby('value')['rectangle_count'].agg(['mean', 'std']).reset_index()
        ax.plot(stats['value'], stats['mean'], 'o-',
                color='steelblue', linewidth=2, markersize=7)
        ax.fill_between(stats['value'],
                        stats['mean'] - stats['std'],
                        stats['mean'] + stats['std'],
                        alpha=0.15, color='steelblue')
        ax.set_xlabel('patience')
        ax.set_ylabel('Mean rectangle count')
        ax.set_xticks(patience_vals)

    plt.tight_layout()
    plt.savefig('fig_patience_quality.png', bbox_inches='tight')
    plt.show()
else:
    print('EXP-2B results not available.')


### EXP-2 – Porovnanie inicializačných metód GA

In [ ]:
INIT_METHODS = ['dm', 'gdm', 'random', 'quadtree', 'largest_rect']
INIT_ALGO_MAP = {
    'dm': 'ga_dm', 'gdm': 'ga_gdm', 'random': 'ga_random',
    'quadtree': 'ga_qtd', 'largest_rect': 'ga_lrf',
}
ANALYSIS_DATASETS = ['analysis/objects_quartile', 'analysis/leafs_quartile']

init_records = []
for method in INIT_METHODS:
    algo = INIT_ALGO_MAP[method]
    for ds in ANALYSIS_DATASETS:
        df = load_results(algo, ds)
        if df.empty:
            continue
        exp1 = df[df['run_path'].str.contains('exp1_init_run2', na=False)]
        if exp1.empty:
            continue
        for _, row in exp1.iterrows():
            init_records.append({
                'init_method': method,
                'dataset': ds.split('/')[-1],
                'image_name': row['image_name'],
                'seed': row.get('seed'),
                'rectangle_count': row['rectangle_count'],
                'execution_time_sec': row['execution_time_sec'],
                'generations_used': row.get('generations_used'),
            })

init_df = pd.DataFrame(init_records)
if not init_df.empty:
    summary = (init_df.groupby(['init_method', 'dataset'])
               .agg(
                   mean_rects=('rectangle_count', 'mean'),
                   std_rects=('rectangle_count', 'std'),
                   mean_time=('execution_time_sec', 'mean'),
                   std_time=('execution_time_sec', 'std'),
                   count=('rectangle_count', 'count'),
               )
               .round(2))
    display(summary)
else:
    print('EXP-1 results not available yet.')
    print('Run: python -m experiments.scripts.analysis.run_ga_exp1_init')

In [ ]:
if not init_df.empty:
    for ds in init_df['dataset'].unique():
        sub = init_df[init_df['dataset'] == ds].copy()

        # Assign quartile labels from manifest pixel counts
        manifest = DATA_DIR / 'analysis' / ds / 'manifest.csv'
        if manifest.exists():
            mf = pd.read_csv(manifest).sort_values('pixels').reset_index(drop=True)
            n = len(mf)
            mf['quartile'] = pd.cut(
                range(n), bins=4,
                labels=['Q1 (malé)', 'Q2', 'Q3', 'Q4 (veľké)'],
            )
            quartile_map = dict(zip(mf['image_name'], mf['quartile']))
            sub['quartile'] = sub['image_name'].map(quartile_map)
            sub = sub.dropna(subset=['quartile'])
        else:
            sub['quartile'] = 'all'

        quartiles = sub['quartile'].cat.categories.tolist() if hasattr(sub['quartile'], 'cat') else sorted(sub['quartile'].unique())
        order = (sub.groupby('init_method')['rectangle_count']
                 .mean().sort_values().index.tolist())

        fig, axes = plt.subplots(1, len(quartiles), figsize=(4 * len(quartiles), 5), sharey=False)
        if len(quartiles) == 1:
            axes = [axes]

        for ax, q in zip(axes, quartiles):
            qsub = sub[sub['quartile'] == q]
            data_by_method = [qsub[qsub['init_method'] == m]['rectangle_count'].values
                              for m in order]
            bp = ax.boxplot(data_by_method, labels=order, patch_artist=True,
                            medianprops={'color': 'black', 'linewidth': 2},
                            showfliers=True,
                            flierprops={'marker': '.', 'markersize': 4, 'alpha': 0.4})
            for patch, color in zip(bp['boxes'], COLORS):
                patch.set_facecolor(color)
                patch.set_alpha(0.8)
            ax.set_yscale('function', functions=(lambda x: np.sqrt(np.maximum(x, 0)), np.square))
            ax.set_ylim(bottom=0)
            ax.set_title(str(q), fontsize=12)
            ax.set_ylabel('Počet obdĺžnikov' if ax == axes[0] else '')
            ax.tick_params(axis='x', rotation=35, labelsize=11)

        fig.suptitle(ds, fontsize=13)
        plt.tight_layout()
        plt.savefig(f'fig_04_ga_init_{ds}.png', bbox_inches='tight')
        plt.show()

In [ ]:
if not init_df.empty:
    for ds in init_df['dataset'].unique():
        sub = init_df[init_df['dataset'] == ds]
        order = (sub.groupby('init_method')['rectangle_count']
                 .mean().sort_values().index.tolist())
        data_by_method = [sub[sub['init_method'] == m]['rectangle_count'].values
                          for m in order]

        fig, ax = plt.subplots(figsize=(7, 5))
        bp = ax.boxplot(data_by_method, labels=order, patch_artist=True,
                        medianprops={'color': 'black', 'linewidth': 2},
                        showfliers=True,
                        flierprops={'marker': '.', 'markersize': 4, 'alpha': 0.4})
        for patch, color in zip(bp['boxes'], COLORS):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        ax.set_yscale('function', functions=(lambda x: np.sqrt(np.maximum(x, 0)), np.square))
        ax.set_ylim(bottom=0)
        # ax.set_title(f'{ds} — celý dataset', fontsize=13)
        ax.set_ylabel('Počet obdĺžnikov', fontsize=14)
        ax.tick_params(axis='x', rotation=30, labelsize=13)
        ax.tick_params(axis='y', labelsize=13)
        ax.grid(axis='y', linestyle='--', alpha=0.7, zorder=0)
        ax.set_axisbelow(True)
        plt.tight_layout()
        plt.savefig(f'fig_04_ga_init_{ds}_overall.png', bbox_inches='tight')
        plt.show()

### EXP-3 – Porovnanie metód crossoveru GA

In [ ]:
CROSSOVER_METHODS = [
    'subset_greedy', 'subset_greedy_relaxed',
    'single_point', 'two_point', 'uniform',
]
BEST_INIT_ALGO = 'ga_dm'  # DM init — väčší priestor pre crossover ako GDM

cross_records = []
for ds in ANALYSIS_DATASETS:
    df = load_results(BEST_INIT_ALGO, ds)
    if df.empty:
        continue
    exp3 = df[df['run_path'].str.contains('exp3_crossover', na=False)]
    if exp3.empty:
        continue
    for _, row in exp3.iterrows():
        method = row['run_path'].split('exp3_crossover/')[-1].split('/')[0]
        cross_records.append({
            'crossover': method,
            'dataset': ds.split('/')[-1],
            'rectangle_count': row['rectangle_count'],
            'execution_time_sec': row['execution_time_sec'],
            'generations_used': row.get('generations_used'),
        })

cross_df = pd.DataFrame(cross_records)
if not cross_df.empty:
    summary = (cross_df.groupby(['crossover', 'dataset'])
               .agg(
                   mean_rects=('rectangle_count', 'mean'),
                   std_rects=('rectangle_count', 'std'),
                   mean_time=('execution_time_sec', 'mean'),
                   std_time=('execution_time_sec', 'std'),
                   n=('rectangle_count', 'count'),
               ).round(2))
    display(summary)
else:
    print('EXP-3 results not available yet.')
    print('Run: python -m experiments.scripts.analysis.run_ga_exp3_crossover')

In [ ]:
if not cross_df.empty:
    for ds in cross_df['dataset'].unique():
        sub = cross_df[cross_df['dataset'] == ds]
        order = (sub.groupby('crossover')['rectangle_count']
                 .mean().sort_values().index.tolist())
        data_by_method = [sub[sub['crossover'] == m]['rectangle_count'].values
                          for m in order]
        fig, ax = plt.subplots(figsize=(10, 7))
        bp = ax.boxplot(data_by_method, labels=order, patch_artist=True,
                        medianprops={'color': 'black', 'linewidth': 2})
        for patch, color in zip(bp['boxes'], COLORS):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        ax.set_ylabel('Rectangle count', fontsize=14)
        ax.tick_params(axis='x', rotation=30, labelsize=16)
        ax.tick_params(axis='y', labelsize=13)
        plt.tight_layout()
        plt.savefig(f'fig_05_ga_crossover_{ds}.png', bbox_inches='tight')
        plt.show()

In [ ]:
if not cross_df.empty:
    METHOD_ORDER = [
        'subset_greedy_relaxed', 'subset_greedy',
        'single_point', 'two_point', 'uniform',
    ]
    METHOD_LABELS = {
        'subset_greedy_relaxed': 'subset_greedy\nrelaxed',
        'subset_greedy':         'subset_greedy',
        'single_point':          'single_point',
        'two_point':             'two_point',
        'uniform':               'uniform'
    }

    for ds in cross_df['dataset'].unique():
        sub = cross_df[(cross_df['dataset'] == ds)]
        # & (cross_df['crossover'] != 'uniform')]
        stats = (sub.groupby('crossover')
                 .agg(mean_rects=('rectangle_count', 'mean'),
                      mean_time=('execution_time_sec', 'mean'))
                 .reindex([m for m in METHOD_ORDER if m in sub['crossover'].unique()]))

        fig, ax = plt.subplots(figsize=(7, 5))
        for i, (method, row) in enumerate(stats.iterrows()):
            color = COLORS[i % len(COLORS)]
            ax.scatter(row['mean_time'], row['mean_rects'],
                       s=160, color=color, zorder=3, edgecolors='white', linewidths=0.8)
            ax.annotate(METHOD_LABELS.get(method, method),
                        (row['mean_time'], row['mean_rects']),
                        textcoords='offset points', xytext=(6, 4), fontsize=12)

        ax.set_xlabel('Priemerný čas behu (s)')
        ax.set_ylabel('Priemerný počet obdĺžnikov')
        plt.tight_layout()
        plt.savefig(f'fig_exp3_crossover_time_quality_{ds}.png', bbox_inches='tight')
        plt.show()


### EXP-4 – Vplyv p_local a p_merge

In [ ]:
mut_records = []
for ds in ANALYSIS_DATASETS:
    df = load_results('ga_dm', ds)
    if df.empty:
        continue
    exp4 = df[df['run_path'].str.contains('exp4_', na=False)].copy()
    if exp4.empty:
        continue
    exp4['exp'] = exp4['run_path'].str.extract(r'(exp4_\w+)')
    exp4['value'] = exp4['run_path'].str.extract(r'exp4_\w+/([\d.]+)').astype(float)
    for _, row in exp4.iterrows():
        mut_records.append({
            'exp': row['exp'],
            'value': row['value'],
            'dataset': ds.split('/')[-1],
            'image_name': row['image_name'],
            'rectangle_count': row['rectangle_count'],
            'execution_time_sec': row['execution_time_sec'],
        })

mut_df = pd.DataFrame(mut_records)
if not mut_df.empty:
    for exp_name in ['exp4_local', 'exp4_merge']:
        sub = mut_df[mut_df['exp'] == exp_name]
        if sub.empty:
            continue
        print(f'\n=== {exp_name} ===')
        summary = (sub.groupby(['value', 'dataset'])
                   .agg(
                       mean_rects=('rectangle_count', 'mean'),
                       std_rects=('rectangle_count', 'std'),
                       mean_time=('execution_time_sec', 'mean'),
                       std_time=('execution_time_sec', 'std'),
                       n=('rectangle_count', 'count'),
                   ).round(2))
        display(summary)
else:
    print('EXP-4 results not available yet.')
    print('Run: python -m experiments.scripts.analysis.run_ga_mutation_analysis')

In [ ]:
if not mut_df.empty:
    exp_map = {
        'exp4_local': ('Pravdepodobnosť lokálnej mutácie (p_local)', 'fig_11_exp4_local.png'),
        'exp4_merge': ('Pravdepodobnosť mutácie zlúčenia (p_merge)',  'fig_11_exp4_merge.png'),
    }
    all_ds = mut_df['dataset'].unique()
    ds_labels = sorted(all_ds, key=lambda d: (0 if 'object' in d else 1))
    FONTSIZE = 15

    for exp_name, (xlabel, fname) in exp_map.items():
        sub_exp = mut_df[mut_df['exp'] == exp_name]
        if sub_exp.empty:
            print(f'No data for {exp_name}.')
            continue

        fig, axes = plt.subplots(1, len(ds_labels),
                                 figsize=(7 * len(ds_labels), 5),
                                 squeeze=False)

        for col_i, ds in enumerate(ds_labels):
            ax = axes[0][col_i]
            sub = sub_exp[sub_exp['dataset'] == ds]
            if sub.empty:
                ax.text(0.5, 0.5, 'Žiadne dáta', ha='center', va='center',
                        transform=ax.transAxes, fontsize=FONTSIZE)
                continue

            stats = sub.groupby('value')['rectangle_count'].agg(['mean', 'std']).reset_index()
            ax.plot(stats['value'], stats['mean'], 'o-', color='steelblue', linewidth=2, markersize=8)
            ax.set_xlabel(xlabel, fontsize=FONTSIZE)
            ax.set_ylabel('Priemerný počet obdĺžnikov', fontsize=FONTSIZE)
            ax.tick_params(labelsize=FONTSIZE - 2)
            ax.grid(True, color='#e0e0e0', linewidth=0.8)
            ax.set_axisbelow(True)

        plt.tight_layout()
        plt.savefig(fname, bbox_inches='tight')
        plt.show()
else:
    print('No mutation analysis data found for exp4_local / exp4_merge.')

## 3. Full Run – Porovnanie algoritmov

Finálne porovnanie na kompletných datasetoch `leafs_selected` (204) a `objects_selected` (282).

## 3. Full Run – Porovnanie algoritmov

In [ ]:
# ── Sledovacia tabuľka – Full Run ───────────────────────────────────
records = []

# Deterministické algoritmy
for algo in DETERMINISTIC_ALGOS:
    for ds in DATASETS_COMPARE:
        total = _dataset_total(ds)
        if algo == 'gdm':
            p = CSV_DIR / algo / ds / 'run2' / 'results.csv'
            df = pd.read_csv(p) if p.exists() else load_baseline(algo, ds)
            if not df.empty:
                n = len(df)
                records.append({
                    'algorithm': algo, 'run_id': 'run1', 'dataset': ds,
                    'progress': f'{n}/{total} ({n/total*100:.0f}%)',
                    'n': n, 'mean_rect': df['rectangle_count'].mean(),
                    'std_rect': df['rectangle_count'].std(),
                    'median_rect': df['rectangle_count'].median(),
                    'mean_time': df['execution_time_sec'].mean(),
                })
        elif algo == 'quadtree':
            for run_id, label in [('run1', 'quadtree_fd'), ('run2', 'quadtree_cov')]:
                p = CSV_DIR / algo / ds / run_id / 'results.csv'
                if not p.exists():
                    continue
                df = pd.read_csv(p)
                n = len(df)
                records.append({
                    'algorithm': label, 'run_id': run_id, 'dataset': ds,
                    'progress': f'{n}/{total} ({n/total*100:.0f}%)',
                    'n': n, 'mean_rect': df['rectangle_count'].mean(),
                    'std_rect': df['rectangle_count'].std(),
                    'median_rect': df['rectangle_count'].median(),
                    'mean_time': df['execution_time_sec'].mean(),
                })
        else:
            df = load_baseline(algo, ds)
            if df.empty:
                continue
            n = len(df)
            run_id = df['run_path'].iloc[0].split('/')[-1] if 'run_path' in df.columns else 'run1'
            records.append({
                'algorithm': algo, 'run_id': run_id, 'dataset': ds,
                'progress': f'{n}/{total} ({n/total*100:.0f}%)',
                'n': n, 'mean_rect': df['rectangle_count'].mean(),
                'std_rect': df['rectangle_count'].std(),
                'median_rect': df['rectangle_count'].median(),
                'mean_time': df['execution_time_sec'].mean(),
            })

# GA algoritmy — z MONITOR_CONFIG
for algo in GA_ALGOS:
    run_specs = MONITOR_CONFIG.get(algo, [])
    if not run_specs:
        continue
    for run_id, label in run_specs:
        for ds in DATASETS_COMPARE:
            total = _dataset_total(ds)
            df = load_monitored(algo, ds, run_id)
            n = len(df) if not df.empty else 0
            row = {
                'algorithm': label, 'run_id': run_id, 'dataset': ds,
                'progress': f'{n}/{total} ({n/total*100:.0f}%)' if total else f'{n}/?',
                'n': n,
            }
            if not df.empty:
                row.update({
                    'mean_rect': df['rectangle_count'].mean(),
                    'std_rect': df['rectangle_count'].std(),
                    'median_rect': df['rectangle_count'].median(),
                    'mean_time': df['execution_time_sec'].mean(),
                })
            else:
                row.update({'mean_rect': None, 'std_rect': None,
                            'median_rect': None, 'mean_time': None})
            records.append(row)

monitor_df = pd.DataFrame(records)
# baseline_df — backward compat pre grafy nižšie (algorithm = label)
baseline_df = monitor_df.copy()

if not monitor_df.empty:
    for ds in DATASETS_COMPARE:
        sub = monitor_df[monitor_df['dataset'] == ds]
        if sub.empty:
            continue
        print(f'\n=== {ds} ===')
        disp = sub[['algorithm', 'run_id', 'progress', 'mean_rect',
                     'std_rect', 'median_rect', 'mean_time']].copy()
        display(disp.round(1))
else:
    print('No results available yet.')


In [ ]:
PLOT_ALGOS = [
    'dm',
    'gdm',
    'quadtree_fd',
    'quadtree_cov',
    'largest_rect',
    'graph_based',
    'ga_dm',
    'ga_gdm',
]

if not baseline_df.empty:
    det_df = baseline_df[baseline_df['algorithm'].isin(PLOT_ALGOS)]

    for ds in det_df['dataset'].unique():
        sub = det_df[det_df['dataset'] == ds].sort_values('mean_rect')
        colors = [COLORS[i % len(COLORS)] for i in range(len(sub))]

        x_cap = min(sub['mean_rect'].max() * 1.3, 4000)

        fig, ax = plt.subplots(figsize=(8, 6))
        bars = ax.barh(
            sub['algorithm'], sub['mean_rect'],
            # xerr=sub['std_rect'].fillna(0),
            color=colors,
            edgecolor='white', alpha=0.85,
        )
        ax.set_xscale('function', functions=(lambda x: np.power(np.maximum(x, 0), 0.4), lambda x: np.power(np.maximum(x, 0), 2.5)))
        for bar, (_, row) in zip(bars, sub.iterrows()):
            ax.text(
                row['mean_rect'] * 1.05,
                bar.get_y() + bar.get_height() / 2,
                f"{row['mean_rect']:.1f}",
                va='center', fontsize=11,
            )
        ax.set_xlim(right=x_cap)
        custom_ticks = [t for t in [0, 250, 500, 750, 1000, 2000, 3000, 4000] if t <= x_cap]
        ax.set_xticks(custom_ticks)
        ax.set_xlabel('Priemerný počet obdĺžnikov', fontsize=12)
        ax.tick_params(axis='both', labelsize=12)
        ax.grid(True, color='#e0e0e0', linewidth=0.8, axis='x')
        ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        fname = f"fig_02_algorithm_comparison_{ds.replace('_selected', '')}.png"
        plt.tight_layout()
        plt.savefig(fname, bbox_inches='tight')
        plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, FixedLocator

if not baseline_df.empty:
    for ds in baseline_df['dataset'].unique():
        sub = baseline_df[baseline_df['dataset'] == ds].copy()

        # Pôvodný rozmer grafu a dizajn
        fig, ax = plt.subplots(figsize=(6, 4))

        # 1. NASTAVENIE STUPNICE (symlog)
        # Lineárna do 1300, logaritmická nad tým (stlačí outlier 3000)
        ax.set_yscale('symlog', linthresh=1300)

        # 2. HUSTEJŠIE ČÍSLA NA Y OSI
        y_ticks = [0, 250, 500, 750, 1000, 1250, 2000, 3000]
        ax.yaxis.set_major_locator(FixedLocator(y_ticks))
        ax.yaxis.set_major_formatter(ScalarFormatter())

        x_max = sub['mean_time'].clip(lower=0.01).max()
        y_max = sub['mean_rect'].max()

        texts = []
        for i, (_, row) in enumerate(sub.iterrows()):
            x = max(row['mean_time'], 0.01)
            y = row['mean_rect']

            # Vykreslenie bodu (pôvodný dizajn)
            ax.scatter(x, y, s=120, color=COLORS[i % len(COLORS)], zorder=3, edgecolors='white', lw=0.5)

            # Pridanie textu do zoznamu pre adjust_text
            texts.append(ax.text(x, y, row['algorithm'], fontsize=9))

        # 3. AUTOMATICKÉ ROZSTRKANIE TEXTU (pôvodná logika)
        adjust_text(texts, ax=ax,
                    expand=(1.3, 1.5),
                    arrowprops=dict(arrowstyle='-', color='#aaaaaa', lw=0.8))

        # Pôvodné nastavenia limitov a popisov
        ax.set_xlim(left=-x_max * 0.05, right=x_max * 1.15)
        # Horný limit nastavíme s rezervou pre adjust_text
        ax.set_ylim(bottom=-50, top=y_max * 1.5)

        ax.set_xlabel('Priemerný čas behu (s)')
        ax.set_ylabel('Priemerný počet obdĺžnikov')

        # Mriežka (pôvodný dizajn)
        ax.grid(True, which="major", color='#e0e0e0', linewidth=0.8)
        ax.set_axisbelow(True)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        fname = f"fig_03_time_vs_quality_{ds.replace('_selected', '')}.png"
        plt.tight_layout()
        plt.savefig(fname, bbox_inches='tight', dpi=300)
        plt.show()

## Súhrnná tabuľka

### GA vs GDM vs GBD – Skóre kvality a časová efektivita

In [ ]:
OUTLIER_GAP_THRESHOLD = 250  # % — GA >5x worse than optimal → excluded from plots

GA_FULL_ALGOS  = {'ga_dm': 'GA-DM', 'ga_gdm': 'GA-GDM'}
GA_PLOT_COLORS = {'ga_dm': 'steelblue', 'ga_gdm': 'mediumpurple'}
GA_MARKERS     = {'ga_dm': 'o', 'ga_gdm': '^'}

for ds in DATASETS_COMPARE:
    gdm_raw = load_baseline('gdm', ds)
    gbd_raw = load_baseline('graph_based', ds)

    if gdm_raw.empty or gbd_raw.empty:
        print(f'[{ds}] Chýbajú dáta (gdm={len(gdm_raw)}, gbd={len(gbd_raw)}).')
        continue

    gdm_df = gdm_raw[['image_name', 'rectangle_count', 'execution_time_sec']].rename(
        columns={'rectangle_count': 'rects_gdm', 'execution_time_sec': 'time_gdm'})
    gbd_df = gbd_raw[['image_name', 'rectangle_count', 'execution_time_sec']].rename(
        columns={'rectangle_count': 'rects_gbd', 'execution_time_sec': 'time_gbd'})

    datasets_per_ga = {}
    for algo, label in GA_FULL_ALGOS.items():
        ga_raw = load_monitored(algo, ds, 'run2-pop15-l6-m5') if algo == 'ga_dm' else load_ga_baseline(algo, ds)
        if ga_raw.empty:
            print(f'[{ds}] Chýbajú dáta pre {algo}.')
            continue
        ga_df = ga_raw[['image_name', 'rectangle_count', 'execution_time_sec']].rename(
            columns={'rectangle_count': 'rects_ga', 'execution_time_sec': 'time_ga'})
        m = gdm_df.merge(gbd_df, on='image_name').merge(ga_df, on='image_name')
        if m.empty:
            continue
        m['gap_from_opt_pct']     = (m['rects_ga']  - m['rects_gbd']) / m['rects_gbd'] * 100
        m['improvement_gdm_pct']  = (m['rects_gdm'] - m['rects_ga'])  / m['rects_gbd'] * 100
        m['gdm_gap_from_opt_pct'] = (m['rects_gdm'] - m['rects_gbd']) / m['rects_gbd'] * 100

        outliers = m[m['gap_from_opt_pct'] > OUTLIER_GAP_THRESHOLD].sort_values('gap_from_opt_pct', ascending=False)
        m_plot = m[m['gap_from_opt_pct'] <= OUTLIER_GAP_THRESHOLD].copy()
        datasets_per_ga[algo] = {'m': m, 'm_plot': m_plot, 'outliers': outliers, 'label': label}

        if not outliers.empty:
            print(f'\n[{ds} / {algo}] Outliery vylúčené z plotov (gap > {OUTLIER_GAP_THRESHOLD}%):')
            for _, r in outliers.iterrows():
                print(f'  {r["image_name"]}: GBD={r["rects_gbd"]:.0f}, GA={r["rects_ga"]:.0f}'
                      f' \u2192 {r["gap_from_opt_pct"]:.0f}%')

    if not datasets_per_ga:
        continue

    # Complexity bins based on GDM (same reference for all variants)
    BIN_LABELS = ['Q1\n(jednoduchý)', 'Q2', 'Q3', 'Q4\n(zložitý)']
    first_algo = next(iter(datasets_per_ga))
    bin_map = dict(zip(
        datasets_per_ga[first_algo]['m']['image_name'],
        pd.qcut(datasets_per_ga[first_algo]['m']['rects_gdm'], q=4, labels=BIN_LABELS)
    ))
    for algo in datasets_per_ga:
        datasets_per_ga[algo]['m_plot']['complexity_bin'] = pd.Categorical(
            datasets_per_ga[algo]['m_plot']['image_name'].map(bin_map),
            categories=BIN_LABELS, ordered=True
        )

    # Spoločný set obrázkov — non-outlier pre VŠETKY GA varianty
    common_images = set.intersection(*[set(d['m_plot']['image_name']) for d in datasets_per_ga.values()])
    common_mps = {
        algo: d['m_plot'][d['m_plot']['image_name'].isin(common_images)].copy()
        for algo, d in datasets_per_ga.items()
    }

    # ── Súhrnná tabuľka: GDM | GBD | GA-DM | GA-GDM ────────────────
    base_mp = common_mps[first_algo]
    summary = base_mp.groupby('complexity_bin', observed=True).agg(
        n=('image_name', 'count'),
        GDM=('rects_gdm', 'mean'),
        GBD=('rects_gbd', 'mean'),
    ).round(1)
    for algo, d in datasets_per_ga.items():
        summary[d['label']] = (
            common_mps[algo].groupby('complexity_bin', observed=True)['rects_ga'].mean().round(1)
        )
    n_out_total = len(datasets_per_ga[first_algo]['m']) - len(common_images)
    print(f'\n=== {ds} \u2014 {len(common_images)} obrázkov (bez {n_out_total} outlierov) ===')
    display(summary)

    # ── Spoločné nastavenia pre bary ─────────────────────────────────
    bin_labels = BIN_LABELS
    n_bins = len(bin_labels)
    n_gas = len(datasets_per_ga)
    width = 0.35
    offsets = [(-width / 2) * (n_gas - 1) + i * width for i in range(n_gas)]
    x = range(n_bins)

    # ── Plot 1: scatter \u2014 vzdialenos\u0165 od optima (GBD) ───────────────
    fig1, ax = plt.subplots(figsize=(8, 5))
    first = datasets_per_ga[first_algo]
    ax.scatter(first['m_plot']['rects_gbd'], first['m_plot']['gdm_gap_from_opt_pct'],
               color='orange', alpha=0.45, s=35, label='GDM', edgecolors='none', marker='s')
    for algo, d in datasets_per_ga.items():
        n_out = len(d['outliers'])
        lbl = d['label'] + (f' (bez {n_out})' if n_out else '')
        ax.scatter(d['m_plot']['rects_gbd'], d['m_plot']['gap_from_opt_pct'],
                   color=GA_PLOT_COLORS[algo], alpha=0.55, s=35,
                   label=lbl, edgecolors='none', marker=GA_MARKERS[algo])
    ax.axhline(0, color='green', linewidth=1.2, linestyle='--', label='Optimum (GBD)')
    ax.set_xlabel('Po\u010det obd\u013ažnikov GBD (optimum)')
    ax.set_ylabel('% nad optimom')
    ax.legend(fontsize=9)
    fig1.tight_layout()
    fig1.savefig(f'fig_ga_scatter_{ds}.png', bbox_inches='tight')
    plt.show()

    # ── Plot 2: priemerné zlep\u0161enie GA vs GDM pod\u013aa kvartilu ─────────
    fig2, ax = plt.subplots(figsize=(7, 5))
    for (algo, d), offset in zip(datasets_per_ga.items(), offsets):
        bin_means = common_mps[algo].groupby('complexity_bin', observed=True)['improvement_gdm_pct'].mean()
        ax.bar([xi + offset for xi in x], bin_means,
               width=width, label=d['label'],
               color=GA_PLOT_COLORS[algo], alpha=0.85, edgecolor='white')
    ax.axhline(0, color='#888', linewidth=1, linestyle='--')
    ax.set_xticks(list(x))
    ax.set_xticklabels(bin_labels)
    ax.set_ylabel('Zlep\u0161enie oproti GDM (% optima)')
    ax.legend(fontsize=9)
    fig2.tight_layout()
    fig2.savefig(f'fig_ga_improvement_{ds}.png', bbox_inches='tight')
    plt.show()
